# Lesson 4 - Extraction

## Start ollama by docker compose

In [1]:
!docker compose up -d ollama

 Container ollama  Running


## Pull Meta-Llama-3.1-8B-Claude-GGUF model from Hugging Face

In [2]:
!docker compose exec ollama ollama pull hf.co/bartowski/Meta-Llama-3.1-8B-Claude-GGUF:Q4_K_M

pulling manifest ⠙ pulling manifest 
pulling e5143516efe0: 100% ▕██████████████████▏ 4.9 GB                         
pulling 783adfd1d253: 100% ▕██████████████████▏  976 B                         
pulling 1a9f0f5ed111: 100% ▕██████████████████▏   22 B                         
pulling d9b87732a16b: 100% ▕██████████████████▏  552 B                         
verifying sha256 digest 
writing manifest 
success 


## Set Ollama environment variables

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Create schema

In [4]:
import re
from typing import Optional
from typing_extensions import Annotated
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field, field_validator

# Initialize the model
model1 = ChatOllama(model="hf.co/bartowski/Meta-Llama-3.1-8B-Claude-GGUF:Q4_K_M", temperature=0)

class Person(BaseModel): # Renamed Schema1 to Person for clarity, but you can keep Schema1 if you prefer
    name: Annotated[Optional[str], None, "The name of the person"]
    hair_color: Annotated[Optional[str], None, "The color of the person's hair if known"]
    # Field to capture the raw height string from the LLM
    raw_height: Annotated[Optional[str], None, "The height of the person in any format (e.g., '6 feet', '1.8 meters', '5\\'10\\\"', '180 cm')"]
    # This will be the computed field for height in meters
    height_in_meters: Annotated[Optional[float], None, "The height of the person measured in meters"] 

    @field_validator('height_in_meters', mode='after')
    @classmethod
    def convert_raw_height_to_meters(cls, v: Optional[float], info) -> Optional[float]:
        # Access the raw_height from the data that has been validated so far
        # info.data will contain 'raw_height' after the LLM fills it
        raw_height_str = info.data.get('raw_height')
        if not raw_height_str:
            return None

        height_str = raw_height_str.lower().strip()
        height_str = re.sub(r'\s+', '', height_str)

        # Handle meters
        meter_match = re.search(r"(\d+(\.\d+)?)\s*meters?", height_str)
        if meter_match:
            try:
                return float(meter_match.group(1))
            except ValueError:
                pass

        # Handle feet and inches (e.g., "6 feet", "5'10\"", "5 ft 10 in")
        feet_match = re.search(r"(\d+)(?:'|feet|ft)(\d*)(?:''|\"|inches|in)?", height_str)
        if feet_match:
            feet = float(feet_match.group(1))
            inches = float(feet_match.group(2)) if feet_match.group(2) else 0
            total_inches = (feet * 12) + inches
            return total_inches * 0.0254 # 1 inch = 0.0254 meters

        # Handle centimeters (e.g., "180 cm", "165 centimeters")
        cm_match = re.search(r"(\d+(\.\d+)?)\s*c(enti)?m(eters?)?", height_str)
        if cm_match:
            try:
                return float(cm_match.group(1)) / 100
            except ValueError:
                pass

        # Handle a single number that might imply meters or feet (less reliable without units)
        single_num_match = re.search(r"(\d+(\.\d+)?)", height_str)
        if single_num_match:
            try:
                val = float(single_num_match.group(1))
                if 1 <= val <= 3: # Assuming it's in meters if it's a small number like 1.8
                    return val
                elif 100 <= val <= 250: # Assuming it's in cm if it's a larger number like 180
                    return val / 100
            except ValueError:
                pass

        return None

promptTemplate1 = ChatPromptTemplate([
    ("system", """You are an expert extraction algorithm. Only extract relevant information from the text. If you do not know the value of an attribute asked to extract, return null for the attribute's value."""),
    ("human", "{text}"),
])

# Use the new Person schema (or Schema1 if you renamed it back)
structured_llm = model1.with_structured_output(Person)

# --- Example Usage ---

user_message_feet = "Alan Smith is 6 feet tall and has blond hair."
prompt_feet = promptTemplate1.invoke(input=user_message_feet)
output_feet = structured_llm.invoke(prompt_feet)
print(f"Feet example: {output_feet.model_dump()}")

user_message_meters = "Jane Doe is 1.75 meters tall and has brown hair."
prompt_meters = promptTemplate1.invoke(input=user_message_meters)
output_meters = structured_llm.invoke(prompt_meters)
print(f"Meters example: {output_meters.model_dump()}")

user_message_feet_inches = "John is 5'10\" and has black hair."
prompt_feet_inches = promptTemplate1.invoke(input=user_message_feet_inches)
output_feet_inches = structured_llm.invoke(prompt_feet_inches)
print(f"Feet and inches example: {output_feet_inches.model_dump()}")

user_message_cm = "Maria is 165 cm tall and has red hair."
prompt_cm = promptTemplate1.invoke(input=user_message_cm)
output_cm = structured_llm.invoke(prompt_cm)
print(f"CM example: {output_cm.model_dump()}")

user_message_no_height = "Sarah has green eyes."
prompt_no_height = promptTemplate1.invoke(input=user_message_no_height)
output_no_height = structured_llm.invoke(prompt_no_height)
print(f"No height example: {output_no_height.model_dump()}")


Feet example: {'name': 'Alan Smith', 'hair_color': 'blond', 'raw_height': '6 feet', 'height_in_meters': 1.8288}
Meters example: {'name': 'Jane Doe', 'hair_color': 'brown', 'raw_height': '1. 75 meters', 'height_in_meters': 1.75}
Feet and inches example: {'name': 'John', 'hair_color': 'black', 'raw_height': '5\'10"', 'height_in_meters': 1.778}
CM example: {'name': 'Maria', 'hair_color': 'red', 'raw_height': '165 cm', 'height_in_meters': 1.65}
No height example: {'name': 'Sarah', 'hair_color': None, 'raw_height': None, 'height_in_meters': None}


In [5]:
# Now, define the 'dataSchema' which contains an array (list) of 'Person' objects.
from typing import List

class Schema2(BaseModel):
    """
    Schema for extracting a list of people.
    """
    # This defines 'people' as a list of 'Person' objects.
    # Field(..., description="...") adds the description, similar to Zod's .describe()
    people: List[Person] = Field(..., description="Extracted data about people")

structured_llm2 = model1.with_structured_output(Schema2);
user_message = "My name is Jeff, my hair is black and i am 6 feet tall. Anna has the same color hair as me."
prompt2 = promptTemplate1.invoke(input=user_message)
output2 = structured_llm2.invoke(prompt2)
print(output2.model_dump_json(indent=2))

{
  "people": [
    {
      "name": "Jeff",
      "hair_color": "black",
      "raw_height": "6 feet",
      "height_in_meters": 1.8288
    },
    {
      "name": "Anna",
      "hair_color": "black",
      "raw_height": null,
      "height_in_meters": null
    }
  ]
}
